In [ ]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, BertTokenizer, BertConfig, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from pathlib import Path
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import TrainerCallback

from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.models.bert.modeling_bert import BertAttention, BertModel, BertSelfAttention, BertLayer, BertPreTrainedModel, BertForSequenceClassification

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available else "cpu"

config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=8,
    hidden_size=512,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)

dataset = load_dataset("RussianNLP/russian_super_glue", "terra", trust_remote_code=True)
test_dataset = dataset['validation']
test_dataloader = DataLoader(test_dataset, 50, 1)
print(test_dataset)

russe_dataset = load_dataset("RussianNLP/russian_super_glue", "russe", trust_remote_code=True)
russe_test_dataset = russe_dataset['validation']
russe_test_dataloader = DataLoader(russe_test_dataset, 50, 1)
print(russe_test_dataset)

loss_fn = nn.CrossEntropyLoss()

Dataset({
    features: ['premise', 'hypothesis', 'idx', 'label'],
    num_rows: 307
})
Dataset({
    features: ['word', 'sentence1', 'sentence2', 'start1', 'start2', 'end1', 'end2', 'gold_sense1', 'gold_sense2', 'idx', 'label'],
    num_rows: 8505
})


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/colab_results/models/russe/hybridbert')
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/colab_results/models/russe/hybridbert
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
bert.encoder.layer.{0, 1, 2, 3, 4, 5, 6, 7}.attention.self.LayerNorm.weight | UNEXPECTED |  | 
bert.encoder.layer.{0, 1, 2, 3, 4, 5, 6, 7}.attention.self.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics import Accuracy

acc = Accuracy(task="binary")
model.eval()

with torch.no_grad():
    for batch in test_dataloader:
        print(batch['premise'], batch['hypothesis'])
        inputs = tokenizer(batch['premise'], batch['hypothesis'], return_tensors="pt", padding=True)
        labels = batch['label']
        outputs = model(**inputs)
        preds = torch.tensor([0 if o[0] > o[1] else 1 for o in outputs.logits])
        acc(preds, labels)
        print(acc)

test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")

In [ ]:
from torchmetrics import Accuracy

acc = Accuracy(task="binary")
model.eval()

with torch.no_grad():
    for batch in russe_test_dataloader:
        combined = []
        for ii2 in range(len(batch['word'])):
            combined.append(f"{batch['word'][ii2]} [SEP] {batch['sentence1'][ii2]}")
        print(combined, batch['sentence2'])
        inputs = tokenizer(combined, batch['sentence2'], return_tensors="pt", padding=True)
        labels = batch['label']
        outputs = model(**inputs)
        preds = torch.tensor([0 if o[0] > o[1] else 1 for o in outputs.logits])
        acc(preds, labels)
        print(acc)

test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")
with open(f'/content/drive/MyDrive/colab_results/russe/prebert//accuracy.txt', 'w', encoding='utf-8') as f:
    f.write(str(test_accuracy))